# Complexity Reference — Python Operation Costs

*Reference — flat lookup, no prose. Open this mid-problem when you need the cost of a specific operation.*

Every cost below is for CPython. "Amortised" means the average over a long sequence of operations; a single call can be worse. For *deriving* the complexity of code you have just written, see `05_complexity_analysis`.

## Quick Index

| Section | Content |
| :--- | :--- |
| §1 | list |
| §2 | dict and set |
| §3 | collections — deque, Counter, defaultdict |
| §4 | heapq |
| §5 | str |
| §6 | sorting and searching |
| §7 | memory footprints |
| §8 | the traps that cost the most |

---
## §1 — list

| Operation | Time | Note |
| :--- | :--- | :--- |
| `lst[i]` — index read or write | O(1) | |
| `lst.append(x)` | O(1) amortised | Occasional resize copies everything |
| `lst.pop()` — from the end | O(1) | |
| `lst.pop(0)` — from the front | **O(n)** | Shifts every element. Use `deque` |
| `lst.insert(i, x)` | O(n) | Shifts the tail |
| `del lst[i]` | O(n) | Same reason |
| `x in lst` | **O(n)** | Linear scan. Use a `set` |
| `lst.index(x)` | O(n) | |
| `lst.count(x)` | O(n) | |
| `len(lst)` | O(1) | Length is stored |
| `lst[a:b]` — slice | O(b − a) | Copies |
| `lst + other` | O(n + m) | Builds a new list |
| `lst.reverse()` | O(n) | In place |
| `lst.sort()` | O(n log n) | Timsort, in place, stable |
| `min(lst)` / `max(lst)` / `sum(lst)` | O(n) | |
| `lst.extend(other)` | O(m) | |
| `[0] * n` | O(n) | |

**The one that decides problems:** `pop(0)` and `insert(0, x)` are O(n). A BFS written with `list.pop(0)` is O(n²) and will time out at `n = 10⁵`. `collections.deque` makes both ends O(1).

---
## §2 — dict and set

| Operation | Average | Worst | Note |
| :--- | :--- | :--- | :--- |
| `d[k]` — read | O(1) | O(n) | Worst case needs adversarial hash collisions |
| `d[k] = v` — write | O(1) amortised | O(n) | Resize on growth |
| `k in d` / `k in s` | O(1) | O(n) | |
| `del d[k]` / `s.discard(x)` | O(1) | O(n) | |
| `d.get(k, default)` | O(1) | O(n) | No exception, no extra lookup |
| `len(d)` | O(1) | O(1) | |
| Iterate `d` / `s` | O(n) | O(n) | Insertion order for `dict` (3.7+) |
| `d.keys()` / `.values()` / `.items()` | O(1) to create | | Views, not copies |
| `s1 & s2` — intersection | O(min(n, m)) | | |
| `s1 \| s2` — union | O(n + m) | | |
| `s1 - s2` — difference | O(n) | | |

Keys must be hashable, so `tuple` works and `list` does not. A frozen set of coordinates as `(r, c)` is the standard `visited` representation in grid problems.

**Practical note.** Treat dict and set lookups as O(1); the O(n) worst case does not occur on LeetCode inputs. The real cost is the constant factor — hashing a long string is proportional to its length, so `s in seen` on 10⁵ strings of length 10³ is O(n · k), not O(n).

---
## §3 — collections

### deque

| Operation | Time |
| :--- | :--- |
| `append` / `appendleft` | O(1) |
| `pop` / `popleft` | O(1) |
| `dq[i]` — index | **O(n)** |
| `len(dq)` | O(1) |
| `x in dq` | O(n) |
| `rotate(k)` | O(k) |

Indexing into a deque is O(n) — it is a doubly linked list of blocks, not an array. Use it for the ends only. This is why the monotonic deque only ever touches `dq[0]` and `dq[-1]`, both of which are O(1).

### Counter

| Operation | Time |
| :--- | :--- |
| `Counter(iterable)` | O(n) |
| `c[k]` | O(1), returns `0` for a missing key |
| `c.most_common()` | O(n log n) |
| `c.most_common(k)` | O(n log k) — uses a heap |
| `c1 + c2`, `c1 - c2`, `c1 & c2` | O(n + m) |
| `c1 == c2` | O(n) — the standard anagram check |

### defaultdict

Same costs as `dict`. `defaultdict(list)` and `defaultdict(int)` are the two that matter — the default is created on *access*, so a bare read inserts a key. If that matters, use `d.get(k)` instead.

---
## §4 — heapq

Python's heap is a **min-heap** over a plain list. For a max-heap, push negated values.

| Operation | Time |
| :--- | :--- |
| `heapq.heappush(h, x)` | O(log n) |
| `heapq.heappop(h)` | O(log n) |
| `h[0]` — peek at the minimum | O(1) |
| `heapq.heapify(lst)` | **O(n)** — not O(n log n) |
| `heapq.heappushpop(h, x)` | O(log n) — cheaper than push then pop |
| `heapq.heapreplace(h, x)` | O(log n) — pop then push |
| `heapq.nlargest(k, it)` / `nsmallest(k, it)` | O(n log k) |
| Building a heap by repeated push | O(n log n) |

Two costs worth remembering: `heapify` on an existing list is **O(n)**, so build the list first and heapify once rather than pushing n times. And "top k of n" is O(n log k), which is why a heap beats sorting when k is small.

Tuples in a heap compare element by element, so `(priority, item)` works — but the tie-breaker must also be comparable. Push `(dist, counter, node)` when nodes are not orderable.

---
## §5 — str

Strings are immutable, which is the source of most string-problem complexity surprises.

| Operation | Time |
| :--- | :--- |
| `s[i]` | O(1) |
| `len(s)` | O(1) |
| `s[a:b]` | O(b − a) — copies |
| `s + t` | **O(n + m)** — builds a new string |
| `s += t` in a loop | **O(n²)** overall |
| `''.join(list_of_str)` | O(total length) |
| `t in s` — substring search | O(n · m) worst, near O(n) typical |
| `s.find(t)` / `s.index(t)` | Same |
| `s.split()` / `s.replace()` | O(n) |
| `s.strip()` / `.lower()` / `.upper()` | O(n) |
| `sorted(s)` | O(n log n) — returns a list |
| `s == t` | O(n) |

**The O(n²) trap:** building a string with `result += char` inside a loop is quadratic, because each `+=` copies the whole accumulated string. Append to a list and `''.join(...)` once at the end.

---
## §6 — sorting and searching

| Operation | Time | Space |
| :--- | :--- | :--- |
| `sorted(it)` | O(n log n) | O(n) — new list |
| `lst.sort()` | O(n log n) | O(n) worst for Timsort's merge buffer |
| `sorted(it, key=f)` | O(n log n) calls to `f` | `f` is called once per element, not per comparison |
| `sorted(it, key=cmp_to_key(f))` | O(n log n) calls to `f` | Much slower constant factor |
| `bisect.bisect_left(lst, x)` | O(log n) | Requires a sorted list |
| `bisect.insort(lst, x)` | **O(n)** | O(log n) to find, O(n) to shift |

Timsort is **stable** — equal elements keep their relative order, which is what makes multi-key sorting work by sorting on the least significant key first. It is also close to O(n) on partially sorted input, which is common in interval problems.

`bisect.insort` looks like an O(log n) insert but is O(n) because of the shift. Maintaining a sorted list under n insertions is O(n²); use a heap if you only need the extremes.

---
## §7 — memory footprints

Memory limits are rarely the binding constraint on LeetCode, but when they bite, this is why.

| Object | Approximate bytes |
| :--- | :--- |
| `int` (small) | 28 |
| `int` (large) | 28 + 4 per 30 bits |
| `float` | 24 |
| `bool` | 28 (`True` / `False` are singletons) |
| `list` of n elements | 56 + 8n, **plus the elements themselves** |
| `dict` with n entries | ≈ 64 + 100n |
| `set` with n entries | ≈ 200 + 60n |
| `str` of length n (ASCII) | 49 + n |
| `tuple` of n elements | 40 + 8n + elements |

**The number that surprises people:** a list of one million Python ints is *not* 4 MB. The list holds a million 8-byte pointers (8 MB) plus a million 28-byte int objects (28 MB) — roughly **36 MB**. Small ints from −5 to 256 are cached and shared, so `[0] * 10**6` is only the 8 MB of pointers; a list of a million *distinct* values is not.

Practical consequences:

- A 10⁴ × 10⁴ 2D DP table is 10⁸ cells and will not fit. Roll the DP down to one or two rows.
- `visited` as a set of `(r, c)` tuples costs far more than a 2D boolean list, or than mutating the grid in place with a sentinel.
- `range(n)` is O(1) memory; `list(range(n))` is not.

---
## §8 — the traps that cost the most

Ordered by how often they turn a correct solution into a TLE.

| Trap | Symptom | Fix |
| :--- | :--- | :--- |
| `list.pop(0)` in a BFS loop | TLE at `n ≥ 10⁴`; correct on small tests | `collections.deque` + `popleft()` |
| `s += char` inside a loop | TLE on long strings only | Accumulate in a list, `''.join` at the end |
| `x in list` inside a loop | TLE; O(n²) hidden in an innocuous line | Convert to a `set` first |
| Slicing inside a loop — `s[i:j]` | TLE; each slice is O(j − i) | Compare indices, or use a rolling hash |
| `sorted()` inside a loop | TLE; O(n² log n) | Sort once outside, or use a heap |
| Repeated `heappush` to build a heap | Passes, but n log n where n was available | `heapify` once |
| `bisect.insort` in a loop | TLE; the shift is O(n) | Heap, or batch and sort once |
| Recomputing `len()` or `max()` per iteration | Constant-factor slowdown | Hoist out of the loop |
| Deep recursion on `n = 10⁵` | `RecursionError`, not a wrong answer | Convert to an iterative stack, or raise the limit |
| Building a full 2D DP table when one row suffices | MLE, or a memory-limit verdict | Roll the DP to one or two rows |

For deriving complexity rather than looking it up, continue to `05_complexity_analysis`.